# Session 11 · pandas, Exploratory Data Analysis & Basic Statistics

**Data Mining 2026/27** · doc. dr. Uroš Godnov · **Mon 21 Dec 2026 · online (Zoom)** · 3 × 45 min + 15 min break

In AI Studio you saw data through operators and result views. Today you interrogate a real dataset with **pandas** — 119,390 hotel bookings — and learn the statistics a data miner needs to decide **which patterns are real, how strong they are, and whether they matter**.

| Part | Time | What we do |
|---|---|---|
| Warm-up | 5 min | recall S10 · poll: how many bookings get canceled? |
| **A — Guided walkthrough** | 80 min | selecting & filtering · new columns · groupby, pivot, crosstab, merge · dates · EDA plots · statistics as evidence |
| ☕ Break | 15 min | |
| **B — Your turn** | 35 min | Adult income: load it correctly, explore it, test it, write hypotheses |
| Wrap-up | 10 min | key takeaways, cheat sheet, exit ticket, what's next |
| 🚀 **C — Advanced track (optional)** | after the wrap-up, or at home | statistics by simulation and pandas like a pro: one method chain, loops vs vectors, bootstrap, permutation test, many tests at once, the same EDA in SQL, Simpson's paradox |

**Learning objectives** — after this session you can:
1. select, filter, sort and create columns with pandas;
2. summarise data with `groupby`/`agg`, `pivot_table` and `crosstab`, and combine tables with `merge`;
3. build and use dates;
4. follow an EDA workflow (structure → quality → distributions → relationships → hypotheses) and draw the standard plots;
5. use descriptive statistics, correlation (Pearson vs Spearman), the χ² test with Cramér's V and the Mann–Whitney test as evidence — and tell *statistically significant* from *practically important*;
6. turn EDA findings into hypotheses for a model.

**On Zoom.** Keep Zoom and this notebook side by side. 🔮 predictions are Zoom polls; questions go to the chat (fast-lane questions start with 🚀). **Lost your connection or your runtime restarted?** Click into the cell where we are and choose **Runtime → Run before** — every cell above runs again and you are back in sync.

| Icon | Meaning |
|---|---|
| 🔮 | **Predict before you run** — answer the poll or note your guess |
| 🧪 | **Try it** — a 2–5 minute exercise during the walkthrough |
| ✏️ | **TODO** — your task in Part B |
| ✅ | **Self-check** — instant feedback (⏳ = not attempted yet) |
| 🛟 | **Rescue cell** — run only if you are stuck, to continue with the next task |
| 🤖 | **AI rule** — try on your own first; then Gemini may *explain* or give a *hint* |
| 🚀 | **Fast lane** (optional) — side quests during Part A and the Part C challenges, for students who already code comfortably; never graded |
| 📈 💻 | **Two lenses** — a one-minute link to what you already know: 📈 economics and statistics, 💻 programming and databases |

> 💾 **File → Save a copy in Drive** before you start.

**Data today** (from the web): Hotel Booking Demand (Antonio, de Almeida & Nunes, 2019) · ISO-3166 country table · Adult income (UCI).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/urosgodnov/dm-2627-python/blob/main/notebooks/s11-pandas-eda-statistics.ipynb)

### 🧭 Two lanes — pick yours

| Lane | For whom | How to work through this notebook |
|---|---|---|
| 🟢 **Core** | everyone — especially if Python is still new to you | Part A with the lecturer → Part B → wrap-up. This is everything the assignment and the project need. |
| 🚀 **Fast lane** | you already write Python comfortably (functions, loops, a bit of pandas) — e.g. you passed the 🚀 check in S00 | Follow Part A, and while we cover basics you know, work on the 🚀 **side quests**. Do Part B quickly (it is still the core of the project), then go on to **Part C · 🚀 Advanced track** after the wrap-up. |

You can switch lanes at any time. Everything marked 🚀 is optional, never graded, and its solutions are in the solutions notebook. Use the 📑 table of contents (left sidebar) to jump between parts.

**Two lenses.** 📈 boxes link the topic to economics and statistics, 💻 boxes to programming and databases. Read the one for your programme, or both. In Part C, 📈 challenges need more statistics and less code; 💻 challenges need more programming. **Strong in statistics but new to Python?** Stay in the 🟢 core for the code, and after the wrap-up try the 🧠 think questions and the 📈 challenges.

## 0 · Warm-up (5 min)

### 🔁 Recall from last session (3 min)

Answer **from memory** first — no scrolling back, no AI. Then run the cell below and compare.

1. A Slovenian CSV export uses `;` between values, `,` as the decimal mark, `.` for thousands and the cp1250 encoding. Which `read_csv` arguments read it correctly?
2. `TotalCharges` in the Telco data arrived as text. How did you turn it into numbers, and why was 0 — not the column mean — the right value for the 11 blanks?
3. What does your `quick_profile(df)` return, and why is it the first thing we run today?

In [ ]:
#@title 👀 Show the answers { display-mode: "form" }
print('1. sep=";", decimal=",", thousands="." and encoding="cp1250". Without thousands=".", a value like 125.300 is silently read as 125.3: no error, just a wrong number.')
print('2. pd.to_numeric(..., errors="coerce") turns text that is not a number into NaN. The 11 blanks were brand-new customers (tenure 0) who had not been billed yet, so 0 is the true value; the mean would invent money.')
print('3. One row per column with dtype, n_missing, pct_missing and n_unique. It is the first look at a new table (types, missing values, constant or ID-like columns) before any analysis. Today it runs on 119,390 hotel bookings in A1.')

> 🔮 **Predict before you run:** **Poll:** what share of these hotel bookings do you think was canceled — 10 %, 20 %, 35 % or 50 %?

## ⚙️ Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

pd.set_option("display.max_columns", 40)
sns.set_theme(style="whitegrid")

def quick_profile(df):
    # Your function from S10: one row per column with type, missing values and distinct values.
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (df.isna().mean() * 100).round(1),
        "n_unique": df.nunique(),
    })

print("pandas", pd.__version__)

In [ ]:
#@title 🧰 Course helpers — run this cell (no need to read it) { display-mode: "form" }
CHECK_RESULTS = {}  # label -> True (correct) / False (not yet) / None (not attempted)

def check(label, test, hint=""):
    """Give instant feedback on an exercise without ever stopping 'Run all'.

    `test` is a function (usually a lambda) that returns True when the answer is right.
    """
    try:
        ok = bool(test())
    except NameError as err:
        CHECK_RESULTS[label] = None
        print(f"⏳ {label}: not attempted yet ({err}).")
        return
    except Exception as err:  # wrong type, wrong shape, missing column, ...
        CHECK_RESULTS[label] = False
        print(f"❌ {label}: checking your answer raised {type(err).__name__}: {err}")
        if hint:
            print(f"   💡 Hint: {hint}")
        return
    CHECK_RESULTS[label] = ok
    if ok:
        print(f"✅ {label}: correct!")
    else:
        print(f"❌ {label}: not yet.")
        if hint:
            print(f"   💡 Hint: {hint}")

def progress(*, return_count=False):
    """Summarise all self-checks run so far; optional 🚀 fast-lane checks are counted separately."""
    core = [v for label, v in CHECK_RESULTS.items() if not label.startswith("🚀")]
    fast = [v for label, v in CHECK_RESULTS.items() if label.startswith("🚀")]
    solved = sum(1 for v in core if v)
    print(f"Solved {solved} of {len(core)} exercises checked so far.")
    if fast:
        print(f"🚀 Fast lane (optional): solved {sum(1 for v in fast if v)} of {len(fast)}.")
    for label, v in CHECK_RESULTS.items():
        print(f"  {'✅' if v else ('⏳' if v is None else '❌')} {label}")
    if return_count:
        return solved

print("Helpers loaded ✔")

---
# Part A · Guided walkthrough — 119,390 hotel bookings

## A1 · Load and first look (7 min)

Two hotels in Portugal (a **City Hotel** in Lisbon and a **Resort Hotel** in the Algarve), arrivals from July 2015 to August 2017. Each row is one booking. Key columns: `is_canceled` (target), `lead_time` (days between booking and arrival), `arrival_date_*`, `stays_in_weekend_nights` / `stays_in_week_nights`, `adults` / `children` / `babies`, `market_segment`, `deposit_type`, `customer_type`, `adr` (average daily rate, €), `total_of_special_requests`, and `reservation_status`.

In [ ]:
HOTELS_URL = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv"
hotels = pd.read_csv(HOTELS_URL)
print(hotels.shape)
hotels.head(3)

In [ ]:
quick_profile(hotels).sort_values("pct_missing", ascending=False).head(6)

`company` is missing for 94 % of bookings and `agent` for 14 % — not because data got lost, but because most guests book **without** a company or an agent. This is *structural* missingness; we will treat it accordingly in S12.

In [ ]:
print(hotels["is_canceled"].value_counts())
print("Cancellation rate:", round(hotels["is_canceled"].mean(), 3))

**37 % is our base rate.** Read every group comparison today against it: a group with a 60 % cancellation rate is risky, one with 15 % is safe.

## A2 · Selecting, filtering and new columns (15 min)

| Task | pandas | AI Studio |
|---|---|---|
| some columns | `df[["a", "b"]]` | Select Attributes |
| rows by condition | `df[df["a"] > 3]`, combine with `&` (and), `\|` (or), `~` (not) | Filter Examples |
| value in a list / range | `df["a"].isin([...])`, `df["a"].between(1, 5)` | Filter Examples |
| readable conditions | `df.query("a > 3 and b == 'x'")` | — |
| rows *and* columns | `df.loc[mask, ["a", "b"]]` | — |
| new column | `df["c"] = df["a"] + df["b"]` | Generate Attributes |

In [ ]:
cols = ["hotel", "is_canceled", "lead_time", "adr", "country", "deposit_type"]
hotels[cols].head()

In [ ]:
long_lead = hotels[hotels["lead_time"] > 365]                                     # booked over a year ahead
resort_pt = hotels[(hotels["hotel"] == "Resort Hotel") & (hotels["country"] == "PRT")]
summer = hotels[hotels["arrival_date_month"].isin(["July", "August"])]
print(len(long_lead), "bookings made more than a year ahead")
print(len(resort_pt), "Resort Hotel bookings by guests from Portugal")
print(len(summer), "summer bookings")

hotels.query("adr > 300 and is_canceled == 0")[cols].sort_values("adr", ascending=False).head()

> ⚠️ **Common mistake:** leaving out the brackets, as in `hotels[hotels["hotel"] == "City Hotel" & hotels["lead_time"] > 365]`. Python applies `&` *before* `==` and `>`, so you get a `TypeError` (or a wrong result). Writing `and` / `or` instead of `&` / `|` raises *"The truth value of a Series is ambiguous"*. The rule: **one pair of brackets around each condition**, joined with `&`, `|` or `~`.

In [ ]:
hotels["total_nights"] = hotels["stays_in_weekend_nights"] + hotels["stays_in_week_nights"]
hotels["total_guests"] = hotels["adults"] + hotels["children"].fillna(0) + hotels["babies"]
hotels["revenue"] = hotels["adr"] * hotels["total_nights"]          # booked value (canceled bookings included)
hotels[["total_nights", "total_guests", "adr", "revenue"]].describe().round(1)

> 🔮 **Predict before you run:** `describe()` shows a **minimum of 0** for `total_guests`. What does that tell you?

In [ ]:
print("Bookings with zero guests:", (hotels["total_guests"] == 0).sum())

The same table hints at more oddities: an `adr` of **−6.4 €**, one of **5,400 €** per night, and bookings with **0 nights** (715 of them). Write them down — EDA *finds* odd values; S12 decides what to do with them.

In [ ]:
# 🧪 Try it: How many City Hotel bookings were for 7 or more nights AND were not canceled?
#    Store the number in `n_city_long`.
# n_city_long = len(hotels[(...) & (...) & (...)])


In [ ]:
#@title ✅ Check — A2 Try it { display-mode: "form" }
check('A2 Try it', lambda: n_city_long == 1823, hint='Three conditions joined with &, each in its own brackets; total_nights >= 7.')

In [ ]:
# 🚀 Side quest: A suspicious column. Make a boolean Series `room_changed` (True when the guest got a different
#    room type than reserved: `assigned_room_type` vs `reserved_room_type`) and store the cancellation
#    rate of both groups in a Series `cancel_by_room` (index False / True). Then cross `room_changed`
#    with `reservation_status`: why must a model NOT use this column to predict cancellations?
# room_changed = ...
# cancel_by_room = ...


In [ ]:
#@title ✅ Check — 🚀 A2 side quest { display-mode: "form" }
check('🚀 A2 side quest', lambda: abs(cancel_by_room.loc[True] - 0.054) < 0.002 and abs(cancel_by_room.loc[False] - 0.416) < 0.002, hint='Compare the two columns with !=, then group is_canceled by that boolean Series.')

## A3 · Aggregating and combining tables (18 min)

`groupby` = **split** the rows into groups → **apply** a function to each group → **combine** the results — the *Aggregate* operator of AI Studio.

In [ ]:
hotels.groupby("hotel")["is_canceled"].mean().round(3)

The City Hotel loses **42 %** of its bookings, the Resort **28 %** — a 14-point gap, answered in one line. (How precise is that gap? Part C, challenge C3, puts a confidence interval around it.)

> 🔮 **Predict before you run:** Which **deposit type** — No Deposit, Non Refund or Refundable — do you expect to have the **lowest** cancellation rate?

In [ ]:
by_deposit = (hotels.groupby("deposit_type")
              .agg(bookings=("is_canceled", "size"),
                   cancel_rate=("is_canceled", "mean"),
                   avg_lead_time=("lead_time", "mean"),
                   avg_adr=("adr", "mean"))
              .round(2))
by_deposit

**Investigate, don't assume.** Notice the second clue in the table: Non Refund bookings were made on average **213 days** ahead, more than twice as early as the rest. Possible explanations: blocks of rooms reserved early by tour operators or groups as "non-refundable" and released in bulk, a booking-system convention, or a labelling problem. A data miner's move is to **ask the hotel** — and to remember that a model trained on this column will learn the oddity, whatever its cause.

In [ ]:
months = ["January", "February", "March", "April", "May", "June", "July",
          "August", "September", "October", "November", "December"]
(hotels.pivot_table(index="arrival_date_month", columns="hotel",
                    values="is_canceled", aggfunc="mean")
 .reindex(months)
 .round(3))

The City Hotel cancels more often in **every** month (37–46 %). At the Resort, cancellations follow the season: from 15 % in January to 33 % in August. A question about two variables at once ("by month *and* hotel") is what `pivot_table` is for.

In [ ]:
pd.crosstab(hotels["market_segment"], hotels["is_canceled"], normalize="index").round(3)

Groups cancel most often (61 %); complimentary, direct and corporate bookings least (13–19 %). The *Undefined* segment shows 100 % — from only **2** bookings: always check the group size before you trust a rate.

> ⚠️ **Common mistake:** reading a crosstab in the wrong direction. `normalize="index"` makes each **row** sum to 1 and answers *"what share of Groups bookings is canceled?"* (61 %). `normalize="columns"` makes each **column** sum to 1 and answers a different question: *"what share of all cancellations comes from Groups?"* (27 %). Say the question out loud before you read the number.

### merge — bringing in a second table

Country codes like `PRT` or `GBR` say little on their own. A public ISO-3166 table maps each code to a country name and a world region. Before merging, always check which keys **will not** find a partner (an *anti-join*):

In [ ]:
ISO_URL = "https://raw.githubusercontent.com/lukes/ISO-3166-Countries-with-Regional-Codes/master/all/all.csv"
iso = pd.read_csv(ISO_URL)[["alpha-3", "name", "region", "sub-region"]]
unmatched = set(hotels["country"].dropna()) - set(iso["alpha-3"])
print("Codes without a match:", unmatched)

> 🔮 **Predict before you run:** After a **left** merge of the bookings with the country table, will the number of rows change?

In [ ]:
hotels["country_iso"] = hotels["country"].replace({"CN": "CHN", "TMP": "TLS"})   # old / 2-letter codes
hotels_geo = hotels.merge(iso, how="left", left_on="country_iso", right_on="alpha-3")
print(hotels.shape, "→", hotels_geo.shape)

(hotels_geo.groupby("region", dropna=False)
 .agg(bookings=("is_canceled", "size"), cancel_rate=("is_canceled", "mean"))
 .sort_values("bookings", ascending=False)
 .round(3))

Same number of rows (only four new columns), so no booking was lost or duplicated. **90 %** of the bookings come from Europe (Portugal alone: 41 %). The `NaN` row collects the 488 bookings without a country plus Taiwan and Antarctica, which have no region in the ISO table — `dropna=False` keeps them visible instead of silently dropping them.

### 🗺️ You already know this

| I want to … | pandas | SQL | Excel | Stata |
|---|---|---|---|---|
| keep some rows and columns | `df.loc[df["adr"] > 100, ["hotel", "adr"]]` | `SELECT hotel, adr FROM t WHERE adr > 100` | AutoFilter, or `=FILTER(…)` | `keep if adr > 100`, then `keep hotel adr` |
| add a column | `df["c"] = df["a"] + df["b"]` | `SELECT *, a + b AS c FROM t` | a formula `=A2+B2`, filled down | `generate c = a + b` |
| sort, largest first | `df.sort_values("adr", ascending=False)` | `… ORDER BY adr DESC` | Data → Sort, or `=SORT(…)` | `gsort -adr` |
| count rows per group | `df["hotel"].value_counts()` | `SELECT hotel, COUNT(*) FROM t GROUP BY hotel` | `=COUNTIF(…)`, or a PivotTable | `tabulate hotel` |
| one summary row per group | `df.groupby("hotel").agg(rate=("is_canceled", "mean"))` | `SELECT hotel, AVG(is_canceled) AS rate FROM t GROUP BY hotel` | PivotTable (Average), or `=AVERAGEIFS(…)` | `collapse (mean) rate=is_canceled, by(hotel)` |
| a two-way table of means | `df.pivot_table(index="month", columns="hotel", values="is_canceled", aggfunc="mean")` | `AVG(CASE WHEN hotel = 'City Hotel' THEN is_canceled END)` per column, `GROUP BY month` | PivotTable with rows, columns and values | `tabulate month hotel, summarize(is_canceled) means` |
| shares within each row | `pd.crosstab(df["a"], df["b"], normalize="index")` | `COUNT(*) * 1.0 / SUM(COUNT(*)) OVER (PARTITION BY a)`, `GROUP BY a, b` | PivotTable → Show Values As → % of Row Total | `tabulate a b, row` |
| add columns from a lookup table | `df.merge(iso, how="left", left_on="country", right_on="alpha-3")` | `… LEFT JOIN iso ON t.country = iso."alpha-3"` | `=XLOOKUP(…)` (or `VLOOKUP`) | `merge m:1 country using iso` |

Find the column of the tool you know. **Stata notes:** `collapse` *replaces* the data in memory by the summary (pandas returns a new table instead); `bysort hotel: egen rate = mean(is_canceled)` adds the group mean as a column and keeps every row. `merge` creates the variable `_merge`, and `tabulate _merge` is the anti-join check from above (`1` = no partner found); `m:1` means many bookings per country, `1:1` a unique key on both sides.

In [ ]:
# 🧪 Try it: Compute the average daily rate (`adr`) per `market_segment`, sorted from highest to lowest.
#    Store the result (a Series) in `adr_by_segment`.
# adr_by_segment = hotels.groupby(...)[...].mean().sort_values(...)


In [ ]:
#@title ✅ Check — A3 Try it { display-mode: "form" }
check('A3 Try it', lambda: adr_by_segment.index[0] == "Online TA" and len(adr_by_segment) == 8, hint='groupby("market_segment")["adr"].mean(), then .sort_values(ascending=False).')

In [ ]:
# 🚀 Side quest: A segment profile in ONE groupby. Build `segment_profile`: one row per `market_segment` with
#    the columns `bookings` (count), `cancel_rate`, `median_lead` (median lead_time) and `share_requests`
#    (share of bookings with at least one special request: a custom function in named aggregation).
#    Keep only segments with at least 1,000 bookings, sorted by cancel_rate (highest first).
# segment_profile = (hotels.groupby("market_segment")
#                    .agg(bookings=(...), cancel_rate=(...), median_lead=(...),
#                         share_requests=("total_of_special_requests", lambda s: ...))
#                    ...)


In [ ]:
#@title ✅ Check — 🚀 A3 side quest { display-mode: "form" }
check('🚀 A3 side quest', lambda: list(segment_profile.index) == ["Groups", "Online TA", "Offline TA/TO", "Corporate", "Direct"] and abs(segment_profile.loc["Online TA", "share_requests"] - 0.633) < 0.002, hint='A lambda inside named aggregation: share_requests=("total_of_special_requests", lambda s: (s > 0).mean()); then .query("bookings >= 1000") on the aggregated table.')

## A4 · Working with dates (8 min)

The arrival date is spread over three columns. Combine them into one real `datetime` column — then pandas can extract weekdays, resample by month and compute differences.

In [ ]:
hotels["arrival_date"] = pd.to_datetime(
    hotels["arrival_date_year"].astype(str) + "-" + hotels["arrival_date_month"] + "-"
    + hotels["arrival_date_day_of_month"].astype(str),
    format="%Y-%B-%d",
)
hotels["arrival_weekday"] = hotels["arrival_date"].dt.day_name()
print(hotels["arrival_date"].min().date(), "→", hotels["arrival_date"].max().date())

> ⚠️ **Common mistake:** letting pandas guess a date format. `pd.to_datetime("03/04/2017")` gives **4 March** (the US order), while a Slovenian file means 3 April. Always pass `format=` (as above) or `dayfirst=True`, and check `min()` / `max()` afterwards.

In [ ]:
monthly = hotels.set_index("arrival_date").resample("MS")["is_canceled"].agg(["size", "mean"])

fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.bar(monthly.index, monthly["size"], width=20, color="tab:blue", alpha=0.6)
ax1.set_xlabel("arrival month")
ax1.set_ylabel("bookings per arrival month")
ax2 = ax1.twinx()
ax2.plot(monthly.index, monthly["mean"], color="tab:red", marker="o")
ax2.set_ylabel("cancellation rate", color="tab:red")
ax1.set_title("Bookings (bars) and cancellation rate (line) by arrival month")
plt.show()

Arrivals drop every winter, and the cancellation rate swings between about **21 %** (November 2015) and **45 %** (July 2015). A model must be able to see this seasonality — S12 turns the arrival month into a feature.

> 📈 **Economist's view.** The arrival date turns 119,390 bookings into a **time series**: seasonality, trend and calendar effects are the same ideas as in the seasonal adjustment of official statistics, or monthly dummies in a regression.

> 💻 **Informatics view.** In a data warehouse this is the **date dimension**: one row per calendar day with year, month, weekday and holiday flags, joined to the fact table by a date key. `.dt` builds the same attributes on the fly.

In [ ]:
weekdays = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
hotels.groupby("arrival_weekday")["adr"].mean().reindex(weekdays).round(1)

Friday and Sunday arrivals pay the most (≈ 105 € per night), Wednesday and Thursday arrivals the least (≈ 99 €): a difference of about 7 €.

In [ ]:
# 🧪 Try it: Which arrival weekday has the highest cancellation rate? Store the rates, ordered Monday → Sunday,
#    in a Series `cancel_by_weekday` (reuse the list `weekdays` from the cell above).
# cancel_by_weekday = hotels.groupby(...)[...].mean().reindex(...)


In [ ]:
#@title ✅ Check — A4 Try it { display-mode: "form" }
check('A4 Try it', lambda: list(cancel_by_weekday.index) == weekdays and cancel_by_weekday.idxmax() == "Thursday", hint='Group by "arrival_weekday", take the mean of is_canceled, then .reindex(weekdays).')

In [ ]:
# 🚀 Side quest: When do guests cancel? For a canceled booking, `reservation_status_date` is the day of the
#    cancellation. For the canceled bookings only, compute `days_before` (a Series: days from that date to
#    `arrival_date`), its median `median_days_before`, and `share_last_week`: the share of cancellations
#    made 7 days or less before arrival.
# canceled_rows = hotels[hotels["is_canceled"] == 1]
# days_before = (canceled_rows["arrival_date"] - pd.to_datetime(...)).dt.days
# median_days_before = ...
# share_last_week = ...


In [ ]:
#@title ✅ Check — 🚀 A4 side quest { display-mode: "form" }
check('🚀 A4 side quest', lambda: abs(median_days_before - 54) < 0.5 and abs(share_last_week - 0.143) < 0.002, hint='pd.to_datetime() the status date, subtract it from arrival_date, take .dt.days; the mean of a boolean Series is a share.')

## A5 · The EDA workflow and the standard plots (12 min)

1. **Structure** — size, types (A1)
2. **Quality** — missing values, duplicates, impossible values (A1–A2)
3. **Distributions** — one variable at a time
4. **Relationships** — each feature against the target
5. **Hypotheses** — what should a model learn? (Part B)

> 📈 **Economist's view.** Step 5 is where an econometric model starts: EDA tells you which regressors to try, in which form (lead time looks monotonic, deposit type needs dummies) and which controls you need (the hotel).

> 💻 **Informatics view.** Step 2 is where **data-quality rules** come from: every odd value you find (0 guests, an `adr` of −6.38 €) becomes an automatic check that runs on each new batch of data before it reaches a model or a dashboard.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(hotels["lead_time"], bins=60, ax=axes[0])
axes[0].set_title("lead_time: right-skewed")
axes[0].set_xlabel("lead time (days)")
axes[0].set_ylabel("number of bookings")
sns.boxplot(x=hotels["adr"], ax=axes[1])
axes[1].set_title("adr: one extreme value squeezes the box")
axes[1].set_xlabel("average daily rate (€ per night)")
plt.show()

Most bookings are made a few weeks ahead, but the tail reaches **737 days** — a reason to prefer the median (A6). In the box plot, one booking at **5,400 €** per night squeezes all the others into a thin line: almost certainly an error, to be handled in S12.

In [ ]:
hotels["lead_bin"] = pd.cut(hotels["lead_time"], bins=[-1, 7, 30, 90, 180, 365, 800],
                            labels=["≤7", "8–30", "31–90", "91–180", "181–365", ">365"])
ax = hotels.groupby("lead_bin", observed=True)["is_canceled"].mean().plot(kind="bar", rot=0, color="tab:red")
ax.set_xlabel("lead time (days)")
ax.set_ylabel("cancellation rate")
ax.set_title("The earlier the booking, the likelier the cancellation")
plt.show()

From **10 %** for bookings made in the last week before arrival to **68 %** for bookings made more than a year ahead: `lead_time` is a strong, *monotonic* signal (the earlier, the riskier).

> 🔮 **Predict before you run:** In the correlation heatmap below, which feature will correlate most strongly (in absolute value) with `is_canceled`?

In [ ]:
num_cols = ["is_canceled", "lead_time", "total_nights", "total_guests", "adr", "previous_cancellations",
            "booking_changes", "required_car_parking_spaces", "total_of_special_requests"]
corr = hotels[num_cols].corr(method="spearman")
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            cbar_kws={"label": "Spearman correlation"})
plt.title("Spearman correlations")
plt.show()

Even the strongest correlation with the target is only moderate (|ρ| ≤ 0.32). No single column predicts cancellations on its own — a model will have to combine many weak signals.

In [ ]:
# 🚀 Side quest: A reusable rate plot. Write `plot_rate(df, col, q=None)`: if `q` is given, first cut the numeric
#    column into `q` equal-SIZED groups with `pd.qcut` (`pd.cut(x, 5)` would make equal-WIDTH bins: try it,
#    the last one holds only 164 bookings). Then draw the cancellation rate per group as a bar chart with a
#    dashed line at the overall rate, a title and axis labels, and return the rates (a Series). Use it:
#    `rates_lead = plot_rate(hotels, "lead_time", q=5)` and `rates_customer = plot_rate(hotels, "customer_type")`.
# def plot_rate(df, col, q=None, target="is_canceled"):
#     groups = pd.qcut(df[col], q=q) if q else df[col]
#     ...
#     return rates


In [ ]:
#@title ✅ Check — 🚀 A5 side quest { display-mode: "form" }
check('🚀 A5 side quest', lambda: len(rates_lead) == 5 and rates_lead.is_monotonic_increasing and len(rates_customer) == 4, hint='pd.qcut(df[col], q=5) makes 5 groups of equal size; return the Series of rates, not the plot.')

## A6 · Statistics as evidence (15 min)

Three questions statistics answers for a data miner: **Is the pattern real or noise? How strong is it? Does it matter?**

### Mean or median?

In [ ]:
for col in ["lead_time", "adr"]:
    s = hotels[col]
    print(f"{col:10s} mean={s.mean():7.1f}   median={s.median():7.1f}   skewness={s.skew():5.2f}")

For skewed variables the mean is pulled towards the long tail; the median describes the "typical" booking better.

### Pearson or Spearman?
**Pearson** measures *linear* association and is sensitive to outliers. **Spearman** correlates the *ranks*: it captures any monotonic relationship and is robust.

> 🔮 **Predict before you run:** Ten almost perfectly linear points; then **one** value is mistyped as −30. How much will Pearson and Spearman change?

In [ ]:
x = np.arange(1, 11, dtype=float)
y = x + np.array([0.5, -0.3, 0.2, -0.4, 0.1, 0.3, -0.2, 0.4, -0.1, 0.0])
y_typo = y.copy()
y_typo[-1] = -30                      # one data-entry error

print("Pearson   clean:", round(stats.pearsonr(x, y).statistic, 3), "  with typo:", round(stats.pearsonr(x, y_typo).statistic, 3))
print("Spearman  clean:", round(stats.spearmanr(x, y).statistic, 3), "  with typo:", round(stats.spearmanr(x, y_typo).statistic, 3))

### Two categorical variables: χ² test + Cramér's V
- **χ² test of independence** — H₀: the two variables are unrelated. A small p-value says the pattern is unlikely to be noise.
- **Cramér's V** — *how strong* the association is: 0 = none, 1 = perfect (rough guide: 0.1 small, 0.3 medium, 0.5 large).

In [ ]:
def cramers_v(x, y):
    # Chi-square test + Cramér's V effect size for two categorical variables.
    table = pd.crosstab(x, y)
    chi2, p, dof, expected = stats.chi2_contingency(table)
    v = np.sqrt(chi2 / (table.to_numpy().sum() * (min(table.shape) - 1)))
    return chi2, p, v

chi2, p, v = cramers_v(hotels["deposit_type"], hotels["is_canceled"])
print(f"deposit_type:    chi2 = {chi2:,.0f}   p = {p:.1e}   Cramér's V = {v:.3f}")

**V = 0.48** — a large association, as the 99 % in A3 suggested. (A printed `p = 0.0e+00` is not exactly zero: the true value is too small for the computer to store. Report it as *p < 0.001*.)

> 🔮 **Predict before you run:** Do guests who **arrive on a weekend** cancel at a different rate? Will the difference be statistically significant — and will it be important?

In [ ]:
hotels["weekend_arrival"] = hotels["arrival_date"].dt.day_of_week >= 5      # 5 = Saturday, 6 = Sunday
chi2, p, v = cramers_v(hotels["weekend_arrival"], hotels["is_canceled"])
print(f"weekend_arrival: p = {p:.4f}   Cramér's V = {v:.3f}")
hotels.groupby("weekend_arrival")["is_canceled"].mean().round(3)

Remember the A4 Try it: per weekday, the rates run from 32 % (Sunday) to 41 % (Thursday). The weekend flag lumps Saturday (39.5 %) together with Sunday (32.3 %) and averages the signal away — the seven weekdays give V = 0.07, eight times more than the flag. *How* you encode a feature matters (S12).

> 📈 **Economist's view.** These are the tests from your statistics course, but with 119,390 rows almost every difference earns its significance stars. Judge the size of the effect instead, as you would report a marginal effect and not just its t-statistic.

> 💻 **Informatics view.** A dashboard that shows a KPI one point higher on weekends is not evidence: the gap may be noise or, with this much data, real but too small to act on. A test plus an effect size is the quality gate before a KPI difference becomes a business rule.

In [ ]:
# 🧪 Try it: Your turn: how strongly is `market_segment` associated with `is_canceled`? Use `cramers_v()`
#    and store Cramér's V in `v_segment`. Small, medium or large?
# chi2_seg, p_seg, v_segment = cramers_v(...)


In [ ]:
#@title ✅ Check — A6 Try it { display-mode: "form" }
check('A6 Try it', lambda: abs(v_segment - 0.267) < 0.005, hint='cramers_v(x, y) returns three values: chi2, p and V.')

### A numeric variable in two groups: Mann–Whitney U
Does the distribution of `lead_time` differ between canceled and kept bookings? The **Mann–Whitney U** test compares the two distributions using ranks, so it needs no normality assumption.

In [ ]:
canceled = hotels.loc[hotels["is_canceled"] == 1, "lead_time"]
kept = hotels.loc[hotels["is_canceled"] == 0, "lead_time"]
u, p = stats.mannwhitneyu(canceled, kept)
print(f"Mann–Whitney p = {p:.1e}")
print(f"median lead time: canceled {canceled.median():.0f} days vs kept {kept.median():.0f} days")
print(f"effect size U / (n1 · n2) = {u / (len(canceled) * len(kept)):.2f}")

The p-value says *real*; the effect size says *how strong*. **U/(n₁·n₂) = 0.69** has a plain meaning: take a random canceled and a random kept booking — in 69 % of such pairs the canceled one was booked earlier (ties count half; 0.5 = no difference). It is exactly the **AUC** you know from AI Studio's *Performance* operator, for a "model" that uses `lead_time` alone.

**From statistics to features.** Strong or medium effects (`deposit_type` V = 0.48, `market_segment` V = 0.27, `lead_time` AUC = 0.69) are promising predictors; negligible ones (`weekend_arrival`, V = 0.009) probably are not, however small their p-value. Ranking features by such scores is called **filter feature selection** — S12 automates it with `SelectKBest`.

> **Before the break:** read the Part B scenario below.

---
## ☕ Break — 15 minutes

---
# Part B · Your turn: who earns more than 50K? (35 min)

**Scenario.** A bank wants to offer a premium credit card to high-income customers. Before anyone builds a model, the marketing team asks you to explore the classic **Adult** extract of the 1994 US Census (Becker & Kohavi, 1996; 32,561 people): which characteristics go together with an income above 50,000 USD?

Run the ✅ check after each task. Stuck for more than 3 minutes? Ask in the chat, try Gemini (*explain*/*hint*), or run the 🛟 rescue cell and continue.

In [ ]:
ADULT_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
ADULT_COLUMNS = ["age", "workclass", "fnlwgt", "education", "education_num", "marital_status",
                 "occupation", "relationship", "race", "sex", "capital_gain", "capital_loss",
                 "hours_per_week", "native_country", "income"]

pd.read_csv(ADULT_URL).head(3)       # a naive first attempt — what is wrong with it?

Three problems: the **first data row became the header**, every text value starts with a **space** (`" State-gov"`), and missing values are coded as **`?`** instead of being empty (the first one is in row 13 of this naive table: its `native_country` is `?`).

In [ ]:
# ✏️ TODO: Task 1 — Load the file correctly into `adult`: no header row (use `header=None` and
#    `names=ADULT_COLUMNS`), strip the spaces after the commas (`skipinitialspace=True`) and treat
#    "?" as missing (`na_values="?"`). Then check which columns have missing values.
# adult = pd.read_csv(ADULT_URL, ...)


In [ ]:
#@title ✅ Check — Task 1 { display-mode: "form" }
check('Task 1', lambda: adult.shape == (32561, 15) and adult["workclass"].isna().sum() == 1836, hint='All three arguments are needed: header=None + names=..., skipinitialspace=True, na_values="?".')

In [ ]:
#@title 🛟 Rescue — run only if you could not finish Task 1 { display-mode: "form" }
adult = pd.read_csv(ADULT_URL, header=None, names=ADULT_COLUMNS,
                    skipinitialspace=True, na_values="?")
print("adult loaded:", adult.shape)

In [ ]:
# ✏️ TODO: Task 2 — Add a boolean column `high_income` (True when income is ">50K"). Store the overall
#    share of high earners in `p_high`, and the share of high earners per `education` level, sorted from
#    highest to lowest, in `rate_by_edu`.
# adult["high_income"] = ...
# p_high = ...
# rate_by_edu = ...


In [ ]:
#@title ✅ Check — Task 2 { display-mode: "form" }
check('Task 2', lambda: abs(p_high - 0.2408) < 0.001 and rate_by_edu.index[0] == "Doctorate", hint='The mean of a boolean column is the share of True values.')

In [ ]:
#@title 🛟 Rescue — run only if you could not finish Task 2 { display-mode: "form" }
adult["high_income"] = adult["income"] == ">50K"
print("high_income column added.")

In [ ]:
# ✏️ TODO: Task 3 — (a) `ct_sex`: a crosstab of `sex` × `high_income`, normalised by row (shares per sex).
#    (b) `pv_hours`: a pivot table with the mean `hours_per_week` for each `sex` (rows) and `high_income` (columns).
# ct_sex = pd.crosstab(..., normalize="index")
# pv_hours = adult.pivot_table(index=..., columns=..., values=..., aggfunc="mean")


In [ ]:
#@title ✅ Check — Task 3 { display-mode: "form" }
check('Task 3', lambda: ct_sex.shape == (2, 2) and abs(ct_sex.loc["Female", True] - 0.109) < 0.002 and pv_hours.shape == (2, 2), hint='normalize="index" makes each row sum to 1.')

**✍️ Your answer** — What does `ct_sex` show — and what would a model trained on these data learn from it?

> *(double-click this cell and write your answer here)*

In [ ]:
#@title 🛟 Rescue — run this if `cramers_v` is not defined (e.g. after a runtime restart) { display-mode: "form" }
def cramers_v(x, y):
    # Chi-square test + Cramér's V effect size for two categorical variables.
    table = pd.crosstab(x, y)
    chi2, p, dof, expected = stats.chi2_contingency(table)
    v = np.sqrt(chi2 / (table.to_numpy().sum() * (min(table.shape) - 1)))
    return chi2, p, v
print("cramers_v() is defined.")

In [ ]:
# ✏️ TODO: Task 4 — Statistics as evidence.
#    (a) Test `occupation` against `high_income` with `cramers_v()` from Part A: store the p-value in `p_occ`
#    and Cramér's V in `v_occ`.  (b) Store the Spearman correlation between `age` and `hours_per_week`
#    in `rho_age_hours`.
# chi2_occ, p_occ, v_occ = cramers_v(...)
# rho_age_hours = ...


In [ ]:
#@title ✅ Check — Task 4 { display-mode: "form" }
check('Task 4', lambda: p_occ < 1e-10 and 0.3 < v_occ < 0.4 and 0.1 < rho_age_hours < 0.2, hint='cramers_v(x, y) returns (chi2, p, v); .corr(other, method="spearman").')

In [ ]:
# ✏️ TODO: Task 5 — Compare `hours_per_week` of high and low earners with the Mann–Whitney U test.
#    Store the p-value in `p_hours`, and the two medians in `med_high` and `med_low`.
# high = adult.loc[adult["high_income"], "hours_per_week"]
# low = ...
# u, p_hours = stats.mannwhitneyu(high, low)
# med_high, med_low = ...


In [ ]:
#@title ✅ Check — Task 5 { display-mode: "form" }
check('Task 5', lambda: p_hours < 1e-10 and med_high == med_low == 40, hint='~ negates a boolean mask; stats.mannwhitneyu(a, b) returns (statistic, p-value).')

**✍️ Your answer** — Both medians are exactly 40 hours, yet the test is highly significant. How can that be?

> *(double-click this cell and write your answer here)*

**✍️ Your answer** — **Task 6 — Hypotheses for a model.** Based on your evidence (with effect sizes!), write three hypotheses that a model predicting `high_income` should confirm.

> *(double-click this cell and write your answer here)*

In [ ]:
# ✏️ ⭐ Stretch: Rank ALL text (categorical) columns of `adult` by their Cramér's V with `high_income`.
#    Store a Series sorted from strongest to weakest in `assoc`. (Leave out `income` and `high_income` themselves!)
# cat_cols = adult.select_dtypes(exclude="number").columns.drop([...])
# assoc = pd.Series({c: ... for c in cat_cols}).sort_values(ascending=False)


In [ ]:
#@title ✅ Check — Stretch { display-mode: "form" }
check('Stretch', lambda: assoc.index[0] == "relationship" and "high_income" not in assoc.index and len(assoc) == 8, hint='A dict comprehension {column: V} turned into a pd.Series; cramers_v(...)[2] is V.')

> 🚀 **Done early?** Join the wrap-up with everyone first, then continue with **Part C · 🚀 Advanced track** right after it (📑 table of contents → Part C): six challenges plus an open one that do statistics by simulation — bootstrap, permutation tests, many tests at once — and push your pandas (and SQL) further.

---
## Wrap-up (10 min)

**Key takeaways**
1. Filtering with boolean masks, `groupby`/`agg`, `pivot_table`, `crosstab` and `merge` answer most business questions in one or two lines — always next to the group sizes.
2. Before a merge, anti-join the keys; after it, compare row counts.
3. EDA is detective work: impossible values (zero guests) and surprising patterns (non-refundable bookings that get canceled) must be investigated before modelling.
4. Skewed data call for medians and Spearman; categorical pairs call for χ² **plus** Cramér's V; two groups of numbers for Mann–Whitney **plus** its effect size.
5. With large samples almost everything is "significant" — effect sizes tell you what matters.

In [ ]:
progress()

### 📋 Cheat sheet — today's essentials

| I want to … | Code |
|---|---|
| filter rows by several conditions | `df[(df["a"] > 3) & (df["b"] == "x")]` |
| write a filter as readable text | `df.query("a > 3 and b == 'x'")` |
| summarise per group with named columns | `df.groupby("g").agg(n=("y", "size"), rate=("y", "mean"))` |
| a two-way table of means | `df.pivot_table(index="r", columns="c", values="y", aggfunc="mean")` |
| shares per row (each row sums to 1) | `pd.crosstab(df["a"], df["b"], normalize="index")` |
| find keys without a partner before a merge | `set(df["key"]) - set(lookup["key"])` |
| add columns from a lookup table | `df.merge(lookup, how="left", on="key")` |
| build dates from text | `pd.to_datetime(s, format="%Y-%B-%d")` |
| bin a numeric column | `pd.cut(df["x"], bins=[0, 10, 50, 100])  # pd.qcut(df["x"], q=5): equal-sized` |
| rank (Spearman) correlations | `df[cols].corr(method="spearman")` |
| χ² test + effect size for two categorical columns | `chi2, p, v = cramers_v(df["a"], df["b"])` |
| compare a numeric column in two groups | `u, p = stats.mannwhitneyu(a, b); effect = u / (len(a) * len(b))` |

> 🧩 **For your project:** run today's routine on your project data before you build any model: `quick_profile` → the target rate → the target by each key feature (`groupby` / `crosstab`, **with group sizes**) → dates and seasonality → a ranking of features by Cramér's V / Spearman → three hypotheses **with effect sizes**. Keep a list of impossible values and surprising patterns to ask the data owner about: it becomes the *data understanding* section of your report.

**✍️ Your answer** — **Exit ticket.** (1) When would you use `pivot_table` instead of `groupby`? (2) Why compare row counts after a merge? (3) A feature has p = 0.002 and Cramér's V = 0.009 — keep it as a predictor?

> *(double-click this cell and write your answer here)*

**Next time — S12 (Wed 23 Dec, on Zoom):** we prepare data for a model — missing values, outliers, new features, leakage — first on house prices, then on these hotel bookings, and package every step into a scikit-learn **pipeline**.

## 🧠 Think it through (optional · no code · 10 min)

Questions for thinking rather than coding, e.g. if you are strong in statistics but new to Python. Discuss them in pairs, or write two or three sentences for each. The solutions notebook has model answers.

1. 📈 The hotel manager says: "Non Refund bookings are canceled 99 % of the time, so let us abolish non-refundable rates and cancellations will drop." What is wrong with this causal conclusion, and what would you need to know first?
2. 📈 Weekend arrivals cancel 1 percentage point less often (36.3 % vs 37.3 %, p ≈ 0.002). Would you base a weekend-specific overbooking policy on this? Use the difference between statistical and economic significance.
3. 💻 Every month the booking system exports a new CSV. Which of today's findings would you turn into automatic data-quality checks, and what should happen when a check fails?
4. 💻 The left merge with the ISO table kept all 119,390 rows. Name two ways a join in a production pipeline can silently corrupt data, and how you would guard against each.

> ✍️ *(double-click this cell and write your answers here)*

---
# Part C · 🚀 Advanced track (optional)

**For:** fast-lane students who finished Part B — or anyone who wants more, at home. Never graded; the assignment and the project do not need it.

**Theme:** Statistics by simulation and pandas like a pro. Part A used pandas line by line and statistics from formulas. Here you write an analysis as one readable chain, replace a slow loop by vectorised code, and let the computer do statistics by **simulation**: a bootstrap confidence interval, a permutation test, and a screening of 51 columns, 40 of them pure noise: which 'findings' survive? Finally, the same EDA in SQL.

| # | Challenge | Time | Level | Lens |
|---|---|---|---|---|
| **C1** | One chain: an imperative analysis rewritten as one method chain | 10 min | 🌶️ | 💻 |
| **C2** | Loops vs vectors: row-wise `apply` vs `np.select` on 119,390 rows | 10 min | 🌶️ | 💻 |
| **C3** | Bootstrap a confidence interval for the City − Resort cancellation gap | 15 min | 🌶️🌶️ | 📈 |
| **C4** | A permutation test from scratch, checked against scipy | 15 min | 🌶️🌶️ | 📈 |
| **C5** | Many tests, few discoveries: χ², noise columns and Benjamini–Hochberg | 20 min | 🌶️🌶️🌶️ | 📈 |
| **C6** | The same EDA in SQL: DuckDB on the DataFrame, with a window function | 15 min | 🌶️🌶️ | 💻 |
| **🏁** | Open: find a confounder or a Simpson's paradox (written answer) | 20 min | open | 📈 |

**Lens:** 📈 more statistics and economics, less code · 💻 more programming and data engineering · 📈💻 both

Each challenge stands on its own. **Run the ⚙️ Part C setup cell first**, then pick any challenge. The usual rules apply: a first own attempt, then Gemini may *explain* or give a *hint*.

Part C uses only the hotel data and the ISO country table from Part A; it needs nothing from Part B. DuckDB (C6) is preinstalled in Colab.

In [ ]:
#@title ⚙️ Part C setup — run this first (rebuilds everything Part C needs) { display-mode: "form" }
if "check" not in globals():  # runtime restarted? then restore the ✅ self-check helpers
    CHECK_RESULTS = {}  # label -> True (correct) / False (not yet) / None (not attempted)

    def check(label, test, hint=""):
        """Give instant feedback on an exercise without ever stopping 'Run all'.

        `test` is a function (usually a lambda) that returns True when the answer is right.
        """
        try:
            ok = bool(test())
        except NameError as err:
            CHECK_RESULTS[label] = None
            print(f"⏳ {label}: not attempted yet ({err}).")
            return
        except Exception as err:  # wrong type, wrong shape, missing column, ...
            CHECK_RESULTS[label] = False
            print(f"❌ {label}: checking your answer raised {type(err).__name__}: {err}")
            if hint:
                print(f"   💡 Hint: {hint}")
            return
        CHECK_RESULTS[label] = ok
        if ok:
            print(f"✅ {label}: correct!")
        else:
            print(f"❌ {label}: not yet.")
            if hint:
                print(f"   💡 Hint: {hint}")

    def progress(*, return_count=False):
        """Summarise all self-checks run so far; optional 🚀 fast-lane checks are counted separately."""
        core = [v for label, v in CHECK_RESULTS.items() if not label.startswith("🚀")]
        fast = [v for label, v in CHECK_RESULTS.items() if label.startswith("🚀")]
        solved = sum(1 for v in core if v)
        print(f"Solved {solved} of {len(core)} exercises checked so far.")
        if fast:
            print(f"🚀 Fast lane (optional): solved {sum(1 for v in fast if v)} of {len(fast)}.")
        for label, v in CHECK_RESULTS.items():
            print(f"  {'✅' if v else ('⏳' if v is None else '❌')} {label}")
        if return_count:
            return solved

    print("Helpers loaded ✔")

# Everything Part C needs: safe to run again at any time.
import time

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

HOTELS_URL = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv"
ISO_URL = "https://raw.githubusercontent.com/lukes/ISO-3166-Countries-with-Regional-Codes/master/all/all.csv"
# The bookings and the country table from Part A: reloaded if they are missing (only the original columns are needed)
if not ("hotels" in globals() and isinstance(hotels, pd.DataFrame) and len(hotels) == 119_390):
    hotels = pd.read_csv(HOTELS_URL)
if not ("iso" in globals() and isinstance(iso, pd.DataFrame) and "alpha-3" in iso.columns):
    iso = pd.read_csv(ISO_URL)[["alpha-3", "name", "region", "sub-region"]]

def cramers_v(x, y):
    # Chi-square test + Cramér's V effect size for two categorical variables.
    table = pd.crosstab(x, y)
    chi2, p, dof, expected = stats.chi2_contingency(table)
    v = np.sqrt(chi2 / (table.to_numpy().sum() * (min(table.shape) - 1)))
    return chi2, p, v

print("Part C ready ✔ | hotels", hotels.shape, "| iso", iso.shape, "| duckdb", duckdb.__version__)

## C1 · One chain 🌶️ (10 min) · 💻

**Why it matters.** Code written as one **method chain** — `df.step1().step2().step3()` — reads from top to bottom like a recipe. It leaves no half-finished tables behind (`df`, `df2`, `df_final`, …) and never triggers a `SettingWithCopyWarning`. Most pandas experts write this way.

**Build** `chained`: rewrite the imperative analysis in the next cell as **one chain** that starts with `hotels`. It must equal `reference`: same rows, order, columns and values.

**Hints:** `.query("is_canceled == 0 and adr > 0")` · `.assign(total_nights=..., revenue=lambda d: ...)` — inside one `assign`, a later column may use an earlier one · named aggregation `.agg(stays=("revenue", "size"), ...)` · `.query()` also filters an aggregated table (SQL's `HAVING`) · `.pipe(add_share, "revenue")` puts your own function into a chain.

In [ ]:
def add_share(d, col):
    # Adds share_pct: each row's share (%) of the column total.
    d = d.copy()
    d["share_pct"] = 100 * d[col] / d[col].sum()
    return d

# The imperative way: stays that happened and were paid -> revenue by hotel and customer type
df_ref = hotels[hotels["is_canceled"] == 0].copy()
df_ref = df_ref[df_ref["adr"] > 0].copy()
df_ref["total_nights"] = df_ref["stays_in_weekend_nights"] + df_ref["stays_in_week_nights"]
df_ref["revenue"] = df_ref["adr"] * df_ref["total_nights"]
summary = df_ref.groupby(["hotel", "customer_type"]).agg(
    stays=("revenue", "size"), revenue=("revenue", "sum"), avg_nights=("total_nights", "mean"))
summary = summary[summary["stays"] >= 1000]
summary = add_share(summary, "revenue")
reference = summary.sort_values("revenue", ascending=False).round(2)
reference

In [ ]:
# 🚀 Challenge: C1 — Rewrite the analysis above as ONE method chain that starts with `hotels`.
#    Store the result in `chained`; it must equal `reference`.
# chained = (hotels
#            .query(...)
#            .assign(total_nights=..., revenue=lambda d: ...)
#            .groupby([...])
#            .agg(stays=(...), revenue=(...), avg_nights=(...))
#            .query(...)
#            .pipe(add_share, "revenue")
#            .sort_values(...)
#            .round(2))


In [ ]:
#@title ✅ Check — 🚀 C1 { display-mode: "form" }
def _test():
    pd.testing.assert_frame_equal(chained, reference)   # raises with the details if anything differs
    return True

check('🚀 C1', _test, hint='Same filters, the same two group keys, the columns stays, revenue, avg_nights and share_pct, sorted by revenue (highest first) and rounded to 2 decimals.')

**✍️ Your answer** — A long chain gives a wrong result. How do you find the step that goes wrong?

> *(double-click this cell and write your answer here)*

## C2 · Loops vs vectors 🌶️ (10 min) · 💻

**Why it matters.** `df.apply(f, axis=1)` looks like pandas, but it is a Python **loop in disguise**: it builds a small Series for every row and calls your function 119,390 times. **Vectorised** code works on whole columns at once, in compiled code — often 100× faster. On big data, that decides whether an analysis takes seconds or hours.

**Build** the same risk labels as `labels_apply` (next cell) with `np.select(conditions, choices, default="low")` and store them in `labels_vec`; time it as `t_vec` (seconds) and compute `speedup = t_apply / t_vec`.

**Hints:** one boolean Series per `if` branch, **in the same order** — `np.select` takes the *first* condition that is True, exactly like `if` / `elif` · combine conditions with `&` · time with `start = time.perf_counter()` … `time.perf_counter() - start`.

In [ ]:
RISK_COLS = ["deposit_type", "previous_cancellations", "lead_time", "total_of_special_requests"]

def risk_label(row):
    # A hand-made risk rule for ONE booking (one row).
    if row["deposit_type"] == "Non Refund":
        return "check deposit"
    if row["previous_cancellations"] > 0:
        return "high"
    if row["lead_time"] > 180 and row["total_of_special_requests"] == 0:
        return "high"
    if row["lead_time"] > 30:
        return "medium"
    return "low"

start = time.perf_counter()
labels_apply = hotels[RISK_COLS].apply(risk_label, axis=1)       # the slow way: one Python call per row
t_apply = time.perf_counter() - start
print(f"row-wise apply: {t_apply:.2f} s for {len(hotels):,} rows")
labels_apply.value_counts()

In [ ]:
# 🚀 Challenge: C2 — Build `labels_vec` with np.select (the same labels as `labels_apply`), time it as `t_vec`
#    and compute `speedup = t_apply / t_vec`.
# start = time.perf_counter()
# conditions = [hotels["deposit_type"] == "Non Refund", ...]
# choices = ["check deposit", ...]
# labels_vec = np.select(conditions, choices, default="low")
# t_vec = ...
# speedup = ...


In [ ]:
#@title ✅ Check — 🚀 C2 { display-mode: "form" }
check('🚀 C2', lambda: len(labels_vec) == len(hotels) and (labels_apply.to_numpy() == np.asarray(labels_vec)).all() and speedup > 5, hint="Same order as the if-branches (np.select picks the FIRST condition that is True); default='low'.")

**✍️ Your answer** — Why is `apply(axis=1)` so slow — and what happens if you put the condition `lead_time > 30` first?

> *(double-click this cell and write your answer here)*

## C3 · Bootstrap a confidence interval 🌶️🌶️ (15 min) · 📈

**Why it matters.** The City Hotel cancels **14 percentage points** more often than the Resort (41.7 % vs 27.8 %, A3). How precise is that number? A **confidence interval** (CI) gives the range of plausible values. The **bootstrap** finds it by simulation, without a formula: resample the data many times *with replacement* and watch how much the statistic moves.

**Build**
1. `diff_obs` — the City − Resort difference in cancellation rate;
2. `ci_normal` — the textbook 95 % CI: `diff_obs ± 1.96 · SE`, with SE = √(p₁(1−p₁)/n₁ + p₂(1−p₂)/n₂);
3. `ci_boot` — a 95 % percentile bootstrap CI from 10,000 resamples, **vectorised** (no loop), with `rng = np.random.default_rng(42)`.

**The trick:** resampling the n bookings of one hotel with replacement and counting the cancellations is exactly one draw from a **Binomial(n, p̂)** distribution. So `rng.binomial(n1, p1, size=10_000) / n1` gives 10,000 bootstrap rates for the City Hotel in about a millisecond — instead of 10,000 × 119,390 ≈ 1.2 billion random row picks. Then take the 2.5th and 97.5th percentiles of the 10,000 differences: `np.percentile(diffs, [2.5, 97.5])`.

In [ ]:
# 🚀 Challenge: C3 — Compute `diff_obs`, `ci_normal` (lower, upper) and `ci_boot` (10,000 binomial resamples per
#    hotel, rng = np.random.default_rng(42)).
# rates = hotels.groupby("hotel")["is_canceled"].agg(["mean", "size"])
# p1, n1 = ...   # City Hotel
# p2, n2 = ...   # Resort Hotel
# diff_obs = ...
# se = np.sqrt(...)
# ci_normal = (diff_obs - 1.96 * se, diff_obs + 1.96 * se)
# rng = np.random.default_rng(42)
# boot_diffs = rng.binomial(n1, p1, size=10_000) / n1 - ...
# ci_boot = np.percentile(boot_diffs, [2.5, 97.5])


In [ ]:
#@title ✅ Check — 🚀 C3 { display-mode: "form" }
check('🚀 C3', lambda: abs(diff_obs - 0.1396) < 0.0005 and abs(ci_normal[0] - 0.1341) < 0.0003 and abs(ci_normal[1] - 0.1452) < 0.0003 and abs(ci_boot[0] - 0.1341) < 0.001 and abs(ci_boot[1] - 0.1452) < 0.001, hint='City minus Resort (a positive number); 10,000 binomial draws per hotel, then np.percentile(diffs, [2.5, 97.5]).')

**✍️ Your answer** — Compare the two intervals. When is the bootstrap worth the effort — and what does this interval *not* tell you?

> *(double-click this cell and write your answer here)*

## C4 · A permutation test from scratch 🌶️🌶️ (15 min) · 📈

**Why it matters.** A **permutation test** needs no formula and no normality assumption. The idea: *if* the groups did not matter, shuffling the group labels would change nothing. So we shuffle many times and count how often chance alone produces a difference at least as large as the observed one. It works for any statistic.

**The question.** Imagine you only had a random sample of 1,000 bookings (next cell). In the sample, canceled bookings pay about 4 € more per night than kept ones. Real, or chance?

**Build**
1. `p_perm` — a two-sided permutation p-value for the difference in mean `adr` (canceled − kept), from 10,000 shuffles, **vectorised**: all shuffles at once as a 10,000 × 1,000 matrix, with `rng = np.random.default_rng(42)`;
2. `p_scipy` — the same test with `stats.permutation_test`. Both should agree to about ±0.01.

**Hints:** `rng.permuted(np.tile(pooled, (10_000, 1)), axis=1)` shuffles every row of the matrix independently; in each row, the first `len(adr_canceled)` values play the canceled group · count `np.abs(perm_diffs) >= abs(observed)` and use (count + 1) / (10,000 + 1) · scipy: `stats.permutation_test((adr_canceled, adr_kept), mean_diff, vectorized=True, n_resamples=9_999, rng=np.random.default_rng(42))`, where `mean_diff(a, b, axis)` returns `a.mean(axis=axis) - b.mean(axis=axis)`.

In [ ]:
sample = hotels.sample(1000, random_state=42)          # pretend this is all the data you have
adr_canceled = sample.loc[sample["is_canceled"] == 1, "adr"].to_numpy()
adr_kept = sample.loc[sample["is_canceled"] == 0, "adr"].to_numpy()
observed = adr_canceled.mean() - adr_kept.mean()
print(f"{len(adr_canceled)} canceled vs {len(adr_kept)} kept bookings · difference in mean adr: {observed:.2f} €")

In [ ]:
# 🚀 Challenge: C4 — Compute `p_perm` (from scratch: 10,000 vectorised shuffles, rng = np.random.default_rng(42))
#    and `p_scipy` (stats.permutation_test). Bonus: plot the 10,000 shuffled differences.
# rng = np.random.default_rng(42)
# pooled = np.concatenate([adr_canceled, adr_kept])
# shuffled = rng.permuted(np.tile(pooled, (10_000, 1)), axis=1)
# n_c = len(adr_canceled)
# perm_diffs = ...
# p_perm = ...
#
# def mean_diff(a, b, axis):
#     return ...
# p_scipy = stats.permutation_test(...).pvalue


In [ ]:
#@title ✅ Check — 🚀 C4 { display-mode: "form" }
check('🚀 C4', lambda: abs(p_perm - 0.16) < 0.02 and abs(p_scipy - 0.16) < 0.02, hint='Two-sided: count shuffles with |difference| >= |observed|, then (count + 1) / (10,000 + 1).')

**✍️ Your answer** — Is the 4 € difference real? What do you conclude from the sample — and what would you conclude from all 119,390 bookings?

> *(double-click this cell and write your answer here)*

## C5 · Many tests, few discoveries 🌶️🌶️🌶️ (20 min) · 📈

**Why it matters.** Test 40 useless columns at p < 0.05 and about 2 of them will look "significant" by pure chance. Feature screening in data mining runs exactly such batteries of tests. The **Benjamini–Hochberg (BH)** procedure controls the **false discovery rate** (FDR): the expected share of false alarms among the columns you call significant.

**Build** a DataFrame `tests` with one row per column of `TEST_COLS` (index = column name). `TEST_COLS` holds the 11 real categorical columns of the hotel data plus 40 columns of pure noise (next cell); test each one against `is_canceled` with `cramers_v()`. The columns of `tests`: `chi2`, `p`, `V`, `p_bh` (BH-adjusted p-values from `stats.false_discovery_control`) and `discovery` (`p_bh < 0.05`); rows sorted by `V`, strongest first. Then count: how many noise columns have a raw p < 0.05? How many survive BH?

**Hints:** `{c: cramers_v(screening[c], screening["is_canceled"]) for c in TEST_COLS}` → `pd.DataFrame(results, index=["chi2", "p", "V"]).T` · `stats.false_discovery_control(tests["p"], method="bh")` · `tests.loc[NOISE_COLS, "p"] < 0.05`.

In [ ]:
REAL_COLS = ["hotel", "arrival_date_month", "meal", "country", "market_segment", "distribution_channel",
             "reserved_room_type", "assigned_room_type", "deposit_type", "customer_type", "reservation_status"]
rng = np.random.default_rng(42)
noise = pd.DataFrame({f"noise_{i + 1:02d}": rng.integers(0, 2 + i % 5, size=len(hotels))   # random codes, 2-6 levels
                      for i in range(40)}, index=hotels.index)
NOISE_COLS = list(noise.columns)
TEST_COLS = REAL_COLS + NOISE_COLS
screening = pd.concat([hotels[REAL_COLS + ["is_canceled"]], noise], axis=1)
print(screening.shape, "→", len(REAL_COLS), "real +", len(NOISE_COLS), "noise columns to test")

In [ ]:
# 🚀 Challenge: C5 — Build `tests` (index = column; columns chi2, p, V, p_bh, discovery), sorted by V (strongest
#    first). How many noise columns have a raw p < 0.05, and how many survive BH?
# results = {c: cramers_v(screening[c], screening["is_canceled"]) for c in TEST_COLS}
# tests = pd.DataFrame(results, index=["chi2", "p", "V"]).T
# tests["p_bh"] = ...
# tests["discovery"] = ...
# tests = tests.sort_values(...)


In [ ]:
#@title ✅ Check — 🚀 C5 { display-mode: "form" }
def _test():
    expected_bh = stats.false_discovery_control(tests["p"].to_numpy(dtype=float), method="bh")
    return (set(tests.index) == set(TEST_COLS) and tests["V"].is_monotonic_decreasing
            and np.allclose(tests["p_bh"].to_numpy(dtype=float), expected_bh)
            and tests.loc[REAL_COLS, "discovery"].all() and not tests.loc[NOISE_COLS, "discovery"].any())

check('🚀 C5', _test, hint="One row per column of TEST_COLS; p_bh = stats.false_discovery_control(tests['p'], method='bh'); discovery = p_bh < 0.05; sort by V, descending.")

**✍️ Your answer** — How many noise columns looked significant before and after BH? Why rank by V rather than by p — and what does the top of your ranking tell you?

> *(double-click this cell and write your answer here)*

## C6 · The same EDA in SQL 🌶️🌶️ (15 min) · 💻

**Why it matters.** In most companies the data lives in a database, and the first EDA runs in SQL. **DuckDB** (you met it in S10) is an SQL engine inside Python that queries a pandas DataFrame directly by its variable name: no server, no copying. Moving between SQL and pandas — and proving that both give the same numbers — is an everyday skill.

**Build** three DataFrames with `duckdb.sql("...").df()`:
1. `sql_deposit` — the `by_deposit` table of A3: one row per `deposit_type` with `bookings`, `cancel_rate` and `avg_lead_time`;
2. `sql_region` — the region table of A3: `hotels` LEFT JOIN `iso`, then one row per `region` with `bookings` and `cancel_rate`. Do not forget the code fix (`CN` → `CHN`, `TMP` → `TLS`);
3. `sql_rank` — a **window function**: within each region, rank the countries with at least 200 bookings by cancellation rate (highest first) and keep the top 3. Columns: `region`, `country` (the ISO name), `bookings`, `cancel_rate` and `rank_in_region`; leave out bookings without a region.

Then verify 1 and 2 against pandas.

**Hints:** a column name with a hyphen needs double quotes: `iso."alpha-3"` · the code fix inside the join: `ON i."alpha-3" = CASE h.country WHEN 'CN' THEN 'CHN' WHEN 'TMP' THEN 'TLS' ELSE h.country END` · `HAVING COUNT(*) >= 200` filters whole groups · `RANK() OVER (PARTITION BY region ORDER BY cancel_rate DESC)` · keep the top 3 with a CTE (`WITH ranked AS (…) SELECT … WHERE rank_in_region <= 3`) or DuckDB's shortcut `QUALIFY rank_in_region <= 3` · `GROUP BY` puts all `NULL` regions into one group, like `dropna=False` in pandas.

In [ ]:
# DuckDB finds the DataFrame `hotels` by its name: SQL on a pandas table, without a server or a copy
duckdb.sql("SELECT COUNT(*) AS bookings, AVG(is_canceled) AS cancel_rate FROM hotels").df()

In [ ]:
# 🚀 Challenge: C6 — Build `sql_deposit`, `sql_region` and `sql_rank` with DuckDB (see the list above) and verify
#    the first two against pandas.
# sql_deposit = duckdb.sql('''
#     SELECT deposit_type, COUNT(*) AS bookings, ... AS cancel_rate, ... AS avg_lead_time
#     FROM hotels
#     GROUP BY deposit_type
# ''').df()
#
# sql_region = duckdb.sql('''
#     SELECT i.region, ...
#     FROM hotels AS h
#     LEFT JOIN iso AS i ON i."alpha-3" = CASE h.country WHEN 'CN' THEN 'CHN' WHEN 'TMP' THEN 'TLS' ELSE h.country END
#     GROUP BY i.region
# ''').df()
#
# sql_rank = duckdb.sql('''
#     WITH by_country AS (
#         SELECT ... HAVING COUNT(*) >= 200
#     )
#     SELECT *, RANK() OVER (PARTITION BY region ORDER BY cancel_rate DESC) AS rank_in_region
#     FROM by_country
#     QUALIFY ...
# ''').df()


In [ ]:
#@title ✅ Check — 🚀 C6 { display-mode: "form" }
def _test():
    ref_dep = hotels.groupby("deposit_type").agg(bookings=("is_canceled", "size"), cancel_rate=("is_canceled", "mean"),
                                                 avg_lead_time=("lead_time", "mean"))
    got_dep = sql_deposit.set_index("deposit_type").loc[ref_dep.index, list(ref_dep.columns)]
    codes = hotels["country"].replace({"CN": "CHN", "TMP": "TLS"})
    geo = hotels.assign(country_code=codes).merge(iso, how="left", left_on="country_code", right_on="alpha-3")
    ref_reg = geo.groupby(geo["region"].fillna("(none)"))["is_canceled"].agg(["size", "mean"])
    got_reg = sql_region.assign(region=sql_region["region"].fillna("(none)")).set_index("region")
    by_c = (geo[geo["region"].notna()].groupby(["region", "name"])["is_canceled"]
            .agg(bookings="size", cancel_rate="mean").reset_index())
    by_c = by_c[by_c["bookings"] >= 200].copy()
    by_c["rank"] = by_c.groupby("region")["cancel_rate"].rank(method="min", ascending=False)
    top = by_c[by_c["rank"] <= 3]
    return (np.allclose(got_dep.to_numpy(float), ref_dep.to_numpy(float))
            and set(got_reg.index) == set(ref_reg.index)
            and np.allclose(got_reg.loc[ref_reg.index, ["bookings", "cancel_rate"]].to_numpy(float), ref_reg.to_numpy(float))
            and set(zip(sql_rank["region"], sql_rank["country"])) == set(zip(top["region"], top["name"]))
            and (sql_rank["rank_in_region"] <= 3).all())

check('🚀 C6', _test, hint='Column names exactly as listed; LEFT JOIN with the CN/TMP fix for sql_region; for sql_rank: an inner join, WHERE region IS NOT NULL, HAVING COUNT(*) >= 200, RANK() OVER (PARTITION BY region ...) <= 3.')

**✍️ Your answer** — When would you do EDA in SQL, and when in pandas? What did the comparison with pandas protect you from?

> *(double-click this cell and write your answer here)*

## 🏁 Open challenge · Find a confounder or a Simpson's paradox (20 min) · 📈

**Why it matters.** A pattern in a table can be produced by a **third variable**. A **confounder** is related to both the feature and the target. In a **Simpson's paradox** it even *reverses* a pattern: true for all bookings together, but the opposite inside every group. Models happily learn confounded patterns — analysts must not.

**Your task.** Find one in the hotel data: compare a cancellation rate (or a mean) *overall* and *within the groups* of a third variable. Show the tables, then describe what you found in 3–4 sentences with numbers.

**Hints:** good third variables are `hotel`, `deposit_type`, `market_segment` and `customer_type`. Good features to try: long stays (7+ nights), weekend arrivals, families with children, guests from Portugal, `adr`. A two-way table does the job — `pivot_table(index=third, columns=feature, values="is_canceled", aggfunc=["mean", "size"])` — and always look at the group sizes.

In [ ]:
# 🚀 🏁 Open challenge: Compare a cancellation rate overall and within the groups of a third variable. Look for a
#    pattern that shrinks, vanishes or reverses inside the groups.
# feature = ...          # a boolean Series, e.g. a condition on one column
# third = "hotel"        # the possible confounder
# print(hotels.groupby(feature)["is_canceled"].mean())                          # overall
# hotels.assign(feature=feature).pivot_table(index=third, columns="feature",
#                                            values="is_canceled", aggfunc=["mean", "size"])


**✍️ Your answer** — What did you find? Describe the pattern overall and within the groups, and explain how the third variable produces the difference.

> *(double-click this cell and write your answer here)*

In [ ]:
progress()

---
## Going further (optional, self-study)

- **More than two groups:** `stats.kruskal(*groups)` (rank-based) or `stats.f_oneway(*groups)` (ANOVA) — e.g. `adr` across market segments.
- **Automated EDA reports** (`ydata-profiling`, `sweetviz`) give a fast overview, but they cannot judge *why* something looks odd — that is your job.
- **Reshaping:** `pd.melt()` (wide → long) and `df.pivot()` (long → wide).
- **Two proportions:** `statsmodels.stats.proportion.proportions_ztest` for questions like "do repeat guests cancel less?".
- **Statistics by simulation:** the bootstrap and permutation tests of Part C work for any statistic; `scipy.stats.bootstrap` and `scipy.stats.permutation_test` do the bookkeeping for you.
- **Reading:** Wes McKinney, *Python for Data Analysis*, 3rd ed., ch. 7, 10 and 11 (free at wesmckinney.com/book).

**Data sources and licences:** Hotel Booking Demand — Antonio, de Almeida & Nunes (2019), *Data in Brief* 22, CC BY 4.0 (via TidyTuesday) · ISO-3166 table — github.com/lukes/ISO-3166-Countries-with-Regional-Codes · Adult — Becker & Kohavi (1996), UCI, CC BY 4.0.